# clef-flash:9b + d1:3b + qwen3-vl-2b on the 4 vision-decision experiments (Colab T4)

**What it proves.** The box (8 CPU, 15 GB RAM, no GPU) cannot load `clef-flash:9b` (11 GB) and only gets through subsets of
the d1/qwen runs. This notebook replays the **exact same 1,870 requests** (same state text, question, options and image
pixels, exported by `build_requests.py`) for the RVL-CDIP experiment (16-way, 640 + controls + order-reversed 64),
the pixel-flip pairs (120 + controls), the screenshot gatekeeper (320 + controls) and typographic injection
(240 attacked + 80 clean, with and without the defense sentence) through local Ollama on a T4. All pre-registered rules
live in each experiment's README; scoring happens on the box (`merge` step in README.md). No paid API.

**Order = priority:** clef-flash:9b → d1:3b → qwen3-vl:2b-instruct → d1-omni:600m. A hard time cap stops cleanly and still zips.

**Estimate (unmeasured on T4):** setup + pulls ~15 min; clef-flash ~1.5 s/req × 1,870 ≈ 47 min; d1 ~0.6 s ≈ 19 min;
qwen ~0.8 s ≈ 25 min; d1-omni ~0.3 s ≈ 10 min → **~2 h ≈ 2.3 CU at the measured ~1.1 CU/h, budgeted 4 CU at 2 CU/h.
Hard cap 150 min (5 CU).** SMOKE mode (3 requests per experiment/tag per model, ~33×4) takes ~10 min incl. pulls of the small models only.

Runtime → Change runtime type → **T4 GPU**. Upload `bundle.zip` (~60 MB) when asked, or put it in Drive at
`MyDrive/tig-colab/clef-flash-t4/bundle.zip` and set `USE_DRIVE=True`.

In [ ]:
import os, sys, time, json, pathlib, subprocess, shutil
SMOKE = True            # True: ~10 min plumbing check. Set False for the full run.
USE_DRIVE = False       # True: read bundle from / checkpoint to MyDrive/tig-colab/clef-flash-t4/
HARD_CAP_MIN = 150      # whole-notebook cap (setup + inference); inference stops 8 min before it
MODELS = ['clef-flash:9b', 'd1:3b', 'qwen3-vl:2b-instruct', 'd1-omni:600m']
if SMOKE: MODELS = ['d1-omni:600m', 'd1:3b']   # small pulls; set to MODELS above to smoke clef too
OLLAMA_VERSION = '0.40.7'  # same version as the box
T0 = time.time(); BASE = pathlib.Path('/content/clef_t4'); BASE.mkdir(exist_ok=True)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout
if 'T4' not in gpu: print('WARNING: not a T4 — CU estimate does not apply')
DRIVE = None
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    DRIVE = pathlib.Path('/content/drive/MyDrive/tig-colab/clef-flash-t4'); DRIVE.mkdir(parents=True, exist_ok=True)

## 1 · Bundle (code + exact request images)

In [ ]:
import zipfile, hashlib
B = pathlib.Path('/content/bundle.zip')
if not B.exists() and DRIVE and (DRIVE / 'bundle.zip').exists(): shutil.copy(DRIVE / 'bundle.zip', B)
if not B.exists():
    from google.colab import files; up = files.upload(); B.write_bytes(up['bundle.zip'])
ROOT = BASE / 'bundle'; ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(B) as z:
    for i in z.infolist(): assert (ROOT / i.filename).resolve().is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
for n, s in json.loads((ROOT / 'BUNDLE_SHA256.json').read_text()).items():
    assert hashlib.sha256((ROOT / n).read_bytes()).hexdigest() == s, n
print(sum(1 for _ in open(ROOT / 'requests.jsonl')), 'requests;', len(list((ROOT / 'img').glob('*.webp'))), 'images')

## 2 · Ollama (pinned) + model pulls

In [ ]:
subprocess.run('apt-get -qq install -y zstd >/dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=%s sh' % OLLAMA_VERSION, shell=True, check=True)
print(subprocess.run(['ollama', '--version'], capture_output=True, text=True).stdout)
env = dict(os.environ, OLLAMA_HOST='127.0.0.1:11434', OLLAMA_NUM_PARALLEL='1', OLLAMA_MAX_LOADED_MODELS='1',
           OLLAMA_CONTEXT_LENGTH='8192', OLLAMA_KEEP_ALIVE='30m')
srv = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open(BASE / 'ollama.log', 'w'), stderr=subprocess.STDOUT)
import requests
for _ in range(60):
    try: requests.get('http://127.0.0.1:11434/api/version', timeout=2); break
    except Exception: time.sleep(1)
for m in MODELS:
    t = time.time(); r = subprocess.run(['ollama', 'pull', m], env=env, capture_output=True, text=True)
    print(m, 'pull rc', r.returncode, f'{time.time()-t:.0f}s', r.stderr[-300:])
print(subprocess.run(['ollama', 'list'], env=env, capture_output=True, text=True).stdout)

## 3 · Run (resumable jsonl per model, hard cap, optional Drive sync)

In [ ]:
OUT = BASE / ('smoke' if SMOKE else 'main'); OUT.mkdir(exist_ok=True)
if DRIVE and (DRIVE / OUT.name).exists():
    for f in (DRIVE / OUT.name).glob('*.jsonl'): shutil.copy(f, OUT / f.name)   # resume
cap = max(1, HARD_CAP_MIN - 8 - (time.time() - T0) / 60)
sync = []
if DRIVE: (DRIVE / OUT.name).mkdir(exist_ok=True); sync = ['--sync', str(DRIVE / OUT.name)]
cmd = [sys.executable, str(ROOT / 'colab_run.py'), '--models', ','.join(MODELS), '--cap-min', f'{cap:.1f}', '--out', str(OUT)] + sync + (['--smoke'] if SMOKE else [])
print(' '.join(cmd)); subprocess.run(cmd, check=False)

## 4 · Quick look (full pre-registered scoring runs on the box after merge)

In [ ]:
import collections
for f in sorted(OUT.glob('*.jsonl')):
    rs = [json.loads(l) for l in open(f)]; ok = [r for r in rs if 'error' not in r]
    print(f'== {f.name}: {len(ok)} ok / {len(rs)-len(ok)} errors')
    acc = collections.defaultdict(list); lat = collections.defaultdict(list)
    for r in ok:
        k = (r['exp'], r['tag']); lat[k].append(r.get('latency_ms') or 0)
        if r.get('label') is not None: acc[k].append(r['pred'] == r['label'])
    for k in sorted(lat):
        a = acc.get(k); l = sorted(lat[k])
        print(f'  {k}: n={len(lat[k])}  acc={sum(a)/len(a):.3f}' if a else f'  {k}: n={len(lat[k])}', f' p50 latency {l[len(l)//2]:.0f} ms')

## 5 · Zip and download → put the zip in `returned/` next to `merge_results.py`

In [ ]:
stamp = time.strftime('%Y%m%d_%H%M'); z = BASE / f'results_clef-flash-t4_{stamp}.zip'
shutil.copy(BASE / 'ollama.log', OUT / 'ollama.log')
(OUT / 'env.json').write_text(json.dumps({'gpu': gpu, 'ollama': OLLAMA_VERSION, 'models': MODELS, 'smoke': SMOKE, 'minutes': (time.time() - T0) / 60}))
shutil.make_archive(str(z)[:-4], 'zip', OUT)
if DRIVE: shutil.copy(z, DRIVE / z.name)
print(z, f'{(time.time()-T0)/60:.1f} min used')
from google.colab import files; files.download(str(z))